<a href="https://colab.research.google.com/github/gaelosvaldor-star/Tercer-paracial-/blob/main/Lineas_de_espera.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Sistema de línea de espera con un servidor

Se considera una estación de servicio a la que los clientes llegan según un proceso de Poisson con función de intensidad $\lambda(t)\ge 0$. Hay un único servidor: al llegar un cliente pasa a servicio si el servidor está libre, o se une a la fila si está ocupado. Cuando el servidor termina de atender a un cliente, toma al que ha esperado más tiempo; si no hay nadie en la fila, permanece libre hasta el siguiente arribo.

El planteamiento general admite una intensidad variable $\lambda(t)$; en esta simulación usamos una tasa constante $\lambda$, que corresponde al modelo $M/M/1$. Simulamos la línea de espera y comparamos los resultados con los valores teóricos:

- **Utilización del servidor:** $\rho = \dfrac{\lambda}{\mu}$
- **Clientes promedio en cola:** $L_q = \dfrac{\lambda^2}{\mu(\mu-\lambda)}$
- **Clientes promedio en el sistema:** $L = L_q + \dfrac{\lambda}{\mu}$
- **Tiempo promedio en cola:** $W_q = \dfrac{L_q}{\lambda}$
- **Tiempo promedio en el sistema:** $W = W_q + \dfrac{1}{\mu}$

In [1]:
import numpy as np
import random as r
from collections import deque

In [2]:
def LineaEspera(T, lamb, mu):
    # --- Estado del sistema ---
    t = 0.0                 # tiempo actual
    n = 0                   # clientes en el sistema
    NA = 0                  # arribos
    ND = 0                  # salidas

    # --- Acumuladores de estadísticas ---
    area_sistema = 0.0      # integral de n(t)
    area_cola = 0.0         # integral de (n-1)+ (clientes en cola)
    tiempo_ocupado = 0.0    # tiempo con el servidor ocupado
    suma_W = 0.0            # suma de tiempos en sistema
    suma_Wq = 0.0           # suma de tiempos en cola
    ultimo_evento = 0.0
    cola = deque()

    # Primer arribo
    tA = -(1 / lamb) * np.log(r.random())
    tD = float("inf")
    llegada_en_servicio = None
    servicio_actual = None

    while True:
        t_evento = min(tA, tD)

        # Acumular áreas hasta T
        if t_evento <= T:
            dt = t_evento - ultimo_evento
            area_sistema += n * dt
            area_cola += max(n - 1, 0) * dt
            if n > 0:
                tiempo_ocupado += dt
            ultimo_evento = t_evento

        # CASO 1: arribo
        if tA < tD and tA <= T:
            t = tA
            NA += 1
            n += 1
            cola.append(t)
            tA = t - (1 / lamb) * np.log(r.random())   # siguiente arribo
            if n == 1:                                 # servidor libre -> iniciar servicio
                llegada_en_servicio = cola.popleft()
                servicio_actual = -(1 / mu) * np.log(r.random())
                tD = t + servicio_actual

        # CASO 2: salida dentro de [0, T]
        elif tD < tA and tD <= T:
            t = tD
            n -= 1
            ND += 1
            W = t - llegada_en_servicio
            Wq = W - servicio_actual
            suma_W += W
            suma_Wq += Wq
            if n > 0:
                llegada_en_servicio = cola.popleft()
                servicio_actual = -(1 / mu) * np.log(r.random())
                tD = t + servicio_actual
            else:
                llegada_en_servicio = None
                servicio_actual = None
                tD = float("inf")

        # CASO 3: vaciar el sistema después de T
        elif min(tA, tD) > T and n > 0:
            t = tD
            n -= 1
            ND += 1
            W = t - llegada_en_servicio
            Wq = W - servicio_actual
            suma_W += W
            suma_Wq += Wq
            if n > 0:
                llegada_en_servicio = cola.popleft()
                servicio_actual = -(1 / mu) * np.log(r.random())
                tD = t + servicio_actual
            else:
                llegada_en_servicio = None
                servicio_actual = None
                tD = float("inf")

        # CASO 4: fin
        elif min(tA, tD) > T and n == 0:
            break

    # --- Resultados de la simulación ---
    rho_sim = tiempo_ocupado / T
    L_sim = area_sistema / T
    Lq_sim = area_cola / T
    W_sim = suma_W / ND
    Wq_sim = suma_Wq / ND

    # --- Resultados teóricos (M/M/1) ---
    rho = lamb / mu
    Lq = lamb**2 / (mu * (mu - lamb))
    L = Lq + rho
    Wq = Lq / lamb
    W = Wq + 1 / mu

    print("========== SIMULACIÓN ==========")
    print("Utilización promedio (ρ):", rho_sim)
    print("Clientes promedio en cola (Lq):", Lq_sim)
    print("Clientes promedio en sistema (L):", L_sim)
    print("Tiempo promedio en cola (Wq):", Wq_sim)
    print("Tiempo promedio en sistema (W):", W_sim)

    print("\n========== TEÓRICO ==========")
    print("ρ =", rho)
    print("Lq =", Lq)
    print("L =", L)
    print("Wq =", Wq)
    print("W =", W)

In [3]:
LineaEspera(T=100000, lamb=4, mu=6)

========== SIMULACIÓN ==========
Utilización promedio (ρ): 0.6643252466837459
Clientes promedio en cola (Lq): 1.3215871467246296
Clientes promedio en sistema (L): 1.9859123934083345
Tiempo promedio en cola (Wq): 0.33049303317358886
Tiempo promedio en sistema (W): 0.4966248577933351

========== TEÓRICO ==========
ρ = 0.6666666666666666
Lq = 1.3333333333333333
L = 2.0
Wq = 0.3333333333333333
W = 0.5


Los valores simulados coinciden estrechamente con los teóricos, lo que valida la simulación de la línea de espera $M/M/1$.

## Sistema con dos servidores en serie

Ahora consideramos dos servidores conectados en serie:

$$\text{Llegadas} \longrightarrow \text{Servidor 1} \longrightarrow \text{Servidor 2} \longrightarrow \text{Salida}$$

Cada cliente pasa primero por el servidor 1 y, al terminar, por el servidor 2. Definimos:

- $n_1$: clientes en la estación 1
- $n_2$: clientes en la estación 2
- $t_A$: instante del próximo arribo
- $t_1$: instante de la próxima salida del servidor 1
- $t_2$: instante de la próxima salida del servidor 2

Empezamos definiendo una función auxiliar que genera tiempos exponenciales (tanto entre arribos como de servicio):

In [7]:
def Exp(tasa):
    U = r.random()
    return -(1/tasa)*np.log(U)

Con esa función construimos la simulación del sistema en serie siguiendo el pseudocódigo del libro, añadiendo el cálculo de las mismas métricas del primer modelo ($\rho$, $L$ y $W$) para cada estación:

In [6]:
def LineaEsperaSerie(T, lamb, mu1, mu2):
    t = 0.0
    NA = 0
    ND = 0
    n1 = 0                  # clientes en la estación 1
    n2 = 0                  # clientes en la estación 2

    tA = Exp(lamb)          # próximo arribo
    t1 = float("inf")       # próxima salida del servidor 1
    t2 = float("inf")       # próxima salida del servidor 2

    # --- Acumuladores ---
    area1 = 0.0
    area2 = 0.0
    ocupado1 = 0.0
    ocupado2 = 0.0
    ultimo_evento = 0.0
    llegadas = []           # tiempo de llegada de cada cliente al sistema
    suma_W = 0.0

    while True:
        t_evento = min(tA, t1, t2)

        # Acumular estadísticas sólo hasta T
        if t_evento <= T:
            dt = t_evento - ultimo_evento
            area1 += n1 * dt
            area2 += n2 * dt
            if n1 > 0:
                ocupado1 += dt
            if n2 > 0:
                ocupado2 += dt
            ultimo_evento = t_evento

        # CASO 1: arribo
        if tA <= t1 and tA <= t2 and tA <= T:
            t = tA
            NA += 1
            n1 += 1
            llegadas.append(t)
            tA = t + Exp(lamb)
            if n1 == 1:
                t1 = t + Exp(mu1)

        # CASO 2: salida del servidor 1 -> entra al servidor 2
        elif t1 < tA and t1 <= t2 and t1 <= T:
            t = t1
            n1 -= 1
            n2 += 1
            t1 = float("inf") if n1 == 0 else t + Exp(mu1)
            if n2 == 1:
                t2 = t + Exp(mu2)

        # CASO 3: salida del servidor 2 -> sale del sistema
        elif t2 < tA and t2 < t1 and t2 <= T:
            t = t2
            ND += 1
            n2 -= 1
            W = t - llegadas.pop(0)
            suma_W += W
            t2 = float("inf") if n2 == 0 else t + Exp(mu2)

        # --- Después de T: ya no se aceptan llegadas, se vacía el sistema ---
        elif min(tA, t1, t2) > T:
            tA = float("inf")
            if t1 < t2:
                t = t1
                n1 -= 1
                n2 += 1
                t1 = float("inf") if n1 == 0 else t + Exp(mu1)
                if n2 == 1:
                    t2 = t + Exp(mu2)
            elif t2 < float("inf"):
                t = t2
                ND += 1
                n2 -= 1
                W = t - llegadas.pop(0)
                suma_W += W
                t2 = float("inf") if n2 == 0 else t + Exp(mu2)
            else:
                break

    # --- Resultados ---
    rho1 = ocupado1 / T
    rho2 = ocupado2 / T
    L1 = area1 / T
    L2 = area2 / T
    L = L1 + L2
    W = suma_W / ND if ND > 0 else 0

    print("========== SIMULACIÓN ==========")
    print("ρ1 =", rho1)
    print("ρ2 =", rho2)
    print("L1 =", L1)
    print("L2 =", L2)
    print("L total =", L)
    print("W =", W)
    print("\nClientes que llegaron:", NA)
    print("Clientes que salieron:", ND)

In [8]:
LineaEsperaSerie(T=10000, lamb=2, mu1=3, mu2=4)

========== SIMULACIÓN ==========
ρ1 = 0.6623664659011453
ρ2 = 0.49585231715081757
L1 = 1.9584421451468152
L2 = 0.9820818614566033
L total = 2.9405240066034186
W = 1.4763659372666669

Clientes que llegaron: 19918
Clientes que salieron: 19918
